# Lab02 - Mật mã học hiện đại
**Nhóm 16 - NT101.R11.2**  
**Hà Nguyễn Hiếu An (24520043), Nguyễn Hữu Nghĩa (24521144)**

# Nhiệm vụ 2.2 - Chế độ hoạt động của AES
## Mục tiêu và quy ước
- Mã hóa đúng chuỗi `UIT_LAB_UIT_LAB_UIT_LAB_UIT_LAB_` bằng AES-ECB và AES-CBC, khóa `1234567890123456`, IV cố định `0000000000000000` (16 byte ASCII, không phải 16 byte số 0).
- Chuỗi dài **32 byte**, gồm hai khối 16 byte giống nhau: `UIT_LAB_UIT_LAB_`. In Hex và đối chiếu từng khối.
- Ví dụ chính **không padding** vì dữ liệu đã đủ khối. Phần mở rộng dùng PKCS#7 cho ECB/CBC.
- ECB: $C_i=E_K(P_i)$. CBC: $C_i=E_K(P_i \oplus C_{i-1})$, với $C_0=IV$.
- Thử toàn bộ chế độ AES trong PyCryptodome: ECB, CBC, CFB, OFB, CTR, OPENPGP, EAX, GCM, CCM, OCB, SIV, KW, KWP. CFB thử phân đoạn 8 và 128 bit.
- KW/KWP dành cho **bọc khóa**, thử riêng bằng dữ liệu khóa. SIV cần khóa dài gấp đôi: dùng khóa 32 byte riêng.

In [ ]:
try:
    import Crypto
    from Crypto.Cipher import AES
    from Crypto.Random import get_random_bytes
    from Crypto.Util.Padding import pad, unpad
except ImportError as error:
    raise ImportError('Thiếu PyCryptodome. Chạy %pip install "pycryptodome>=3.23,<4" trong notebook rồi khởi động lại kernel.') from error

if not all(hasattr(AES, name) for name in ('MODE_KW', 'MODE_KWP')):
    raise RuntimeError('Cần PyCryptodome >= 3.23 để thử KW/KWP. Nâng cấp bằng %pip install --upgrade pycryptodome.')

AES_KEY = b'1234567890123456'
AES_SIV_KEY = b'1234567890123456abcdefghijklmnop'
AES_IV = b'0000000000000000'
AES_PLAINTEXT = b'UIT_LAB_UIT_LAB_UIT_LAB_UIT_LAB_'
AES_AAD = b'NT101|Lab02|Task2.2'
AES_NONCES = {
    'CTR': b'ctr-demo',
    'EAX': b'eax-demo-nonce16',
    'GCM': b'gcm-demo-0123',
    'CCM': b'ccm-demo-01',
    'OCB': b'ocb-demo-nonce1',
    'SIV': b'siv-demo-nonce16',
}
AES_MODES = {
    'ECB': AES.MODE_ECB, 'CBC': AES.MODE_CBC,
    'CFB8': AES.MODE_CFB, 'CFB128': AES.MODE_CFB,
    'OFB': AES.MODE_OFB, 'CTR': AES.MODE_CTR, 'OPENPGP': AES.MODE_OPENPGP,
    'EAX': AES.MODE_EAX, 'GCM': AES.MODE_GCM, 'CCM': AES.MODE_CCM,
    'OCB': AES.MODE_OCB, 'SIV': AES.MODE_SIV,
}
AES_IV_MODES = ('CBC', 'CFB8', 'CFB128', 'OFB', 'OPENPGP')
AES_AUTH_MODES = ('EAX', 'GCM', 'CCM', 'OCB', 'SIV')
AES_NONCE_MODES = ('CTR',) + AES_AUTH_MODES


def demo_key(mode):
    return AES_SIV_KEY if mode == 'SIV' else AES_KEY


def demo_options(mode):
    if mode in AES_IV_MODES:
        return {'iv': AES_IV}
    if mode in AES_NONCE_MODES:
        return {'nonce': AES_NONCES[mode]}
    return {}


print('PyCryptodome:', Crypto.__version__)
print('Kích thước khối AES:', AES.block_size, 'byte')
print('Số byte bản rõ:', len(AES_PLAINTEXT))

PyCryptodome: 3.24.1
Kích thước khối AES: 16 byte
Số byte bản rõ: 32
Khóa/IV/nonce cố định dưới đây chỉ dùng thí nghiệm.


## Hàm xử lý
Mỗi lần mã hóa/giải mã tạo cipher mới. Gói kết quả giữ ciphertext, IV/nonce, tag, AAD và quy ước padding, không lưu khóa. OPENPGP tách tiền tố IV mã hóa 18 byte khỏi ciphertext dữ liệu. AEAD dùng `decrypt_and_verify`

In [ ]:
def encrypt_aes(data, key, mode, *, iv=None, nonce=None, padding=False, aad=b''):
    if not isinstance(data, bytes) or not isinstance(key, bytes) or not isinstance(aad, bytes):
        raise ValueError('Dữ liệu, khóa và AAD phải có kiểu bytes.')
    if mode not in AES_MODES:
        raise ValueError('Chế độ AES không được hỗ trợ trong hàm mã hóa văn bản.')
    if not isinstance(padding, bool):
        raise ValueError('padding phải là True hoặc False.')
    if padding and mode not in ('ECB', 'CBC'):
        raise ValueError('PKCS#7 chỉ dùng cho ECB/CBC trong notebook này.')
    if iv is not None and mode not in AES_IV_MODES:
        raise ValueError('Chế độ này không dùng IV.')
    if nonce is not None and mode not in AES_NONCE_MODES:
        raise ValueError('Chế độ này không dùng nonce.')
    if aad and mode not in AES_AUTH_MODES:
        raise ValueError('Chế độ cổ điển không xác thực AAD.')
    payload = pad(data, AES.block_size) if padding else data
    if mode in ('ECB', 'CBC') and len(payload) % AES.block_size:
        raise ValueError('ECB/CBC cần dữ liệu dài bội số 16 byte hoặc padding=True.')
    options = {}
    if mode in AES_IV_MODES and iv is not None:
        options['iv'] = iv
    if mode in AES_NONCE_MODES and nonce is not None:
        options['nonce'] = nonce
    if mode in ('CFB8', 'CFB128'):
        options['segment_size'] = 8 if mode == 'CFB8' else 128
    cipher = AES.new(key, AES_MODES[mode], **options)
    packet = {'mode': mode, 'padding': padding, 'aad': aad}
    if mode in AES_AUTH_MODES:
        cipher.update(aad)
        packet['ciphertext'], packet['tag'] = cipher.encrypt_and_digest(payload)
    elif mode == 'OPENPGP':
        wire = cipher.encrypt(payload)
        packet['encrypted_iv'] = wire[:AES.block_size + 2]
        packet['ciphertext'] = wire[AES.block_size + 2:]
    else:
        packet['ciphertext'] = cipher.encrypt(payload)
    if mode in AES_IV_MODES and mode != 'OPENPGP':
        packet['iv'] = cipher.iv
    if mode in AES_NONCE_MODES and hasattr(cipher, 'nonce'):
        packet['nonce'] = cipher.nonce
    return packet


def decrypt_aes(packet, key):
    mode = packet['mode']
    if mode not in AES_MODES:
        raise ValueError('Chế độ AES không được hỗ trợ.')
    options = {}
    if mode == 'OPENPGP':
        options['iv'] = packet['encrypted_iv']
    elif mode in AES_IV_MODES:
        options['iv'] = packet['iv']
    if mode in AES_NONCE_MODES and 'nonce' in packet:
        options['nonce'] = packet['nonce']
    if mode in ('CFB8', 'CFB128'):
        options['segment_size'] = 8 if mode == 'CFB8' else 128
    cipher = AES.new(key, AES_MODES[mode], **options)
    if mode in AES_AUTH_MODES:
        cipher.update(packet['aad'])
        plaintext = cipher.decrypt_and_verify(packet['ciphertext'], packet['tag'])
    else:
        plaintext = cipher.decrypt(packet['ciphertext'])
    return unpad(plaintext, AES.block_size) if packet['padding'] else plaintext

In [ ]:
def split_blocks(data, block_size=AES.block_size):
    if not isinstance(data, bytes):
        raise ValueError('Dữ liệu tách khối phải là bytes.')
    if not isinstance(block_size, int) or isinstance(block_size, bool) or block_size <= 0:
        raise ValueError('Kích thước khối phải là số nguyên dương.')
    return [data[index:index + block_size] for index in range(0, len(data), block_size)]


def print_blocks(data, label):
    print(f'{label}: {len(data)} byte; Hex = {data.hex()}')
    for index, block in enumerate(split_blocks(data), start=1):
        print(f'  Khối {index} ({len(block):2} byte): {block.hex()}')


def flip_first_bit(data):
    if not data:
        raise ValueError('Cần dữ liệu không rỗng để đảo bit.')
    return bytes([data[0] ^ 1]) + data[1:]


def show_packet(packet, key, original):
    mode = packet['mode']
    print(f'\n--- AES-{mode} ---')
    for field in ('iv', 'nonce', 'encrypted_iv', 'tag'):
        if field in packet:
            print(f'{field}: {packet[field].hex()}')
    if packet['aad']:
        print('AAD:', packet['aad'])
    print_blocks(packet['ciphertext'], 'Ciphertext dữ liệu')
    blocks = split_blocks(packet['ciphertext'])
    if len(blocks) >= 2 and len(blocks[0]) == len(blocks[1]) == AES.block_size:
        print('C1 == C2:', blocks[0] == blocks[1])
    recovered = decrypt_aes(packet, key)
    if recovered != original:
        raise RuntimeError(f'Giải mã AES-{mode} không khớp.')
    print('Giải mã khớp bản rõ:', recovered == original)


def wrap_key(data, wrapping_key, mode):
    if mode not in ('KW', 'KWP'):
        raise ValueError('Chế độ bọc khóa phải là KW hoặc KWP.')
    return AES.new(wrapping_key, getattr(AES, 'MODE_' + mode)).seal(data)


def unwrap_key(ciphertext, wrapping_key, mode):
    if mode not in ('KW', 'KWP'):
        raise ValueError('Chế độ bọc khóa phải là KW hoặc KWP.')
    return AES.new(wrapping_key, getattr(AES, 'MODE_' + mode)).unseal(ciphertext)

## Thí nghiệm chính: ECB và CBC
In bản rõ, ciphertext Hex và từng khối 16 byte; so sánh hai khối ciphertext, kiểm tra giải mã khớp dữ liệu gốc. Không padding ở thí nghiệm này.

In [ ]:
print('Bản rõ:', AES_PLAINTEXT)
print_blocks(AES_PLAINTEXT, 'Bản rõ')
aes_plaintext_blocks = split_blocks(AES_PLAINTEXT)
print('P1 == P2:', aes_plaintext_blocks[0] == aes_plaintext_blocks[1])

aes_ecb_packet = encrypt_aes(AES_PLAINTEXT, AES_KEY, 'ECB')
aes_cbc_packet = encrypt_aes(AES_PLAINTEXT, AES_KEY, 'CBC', iv=AES_IV)
show_packet(aes_ecb_packet, AES_KEY, AES_PLAINTEXT)
show_packet(aes_cbc_packet, AES_KEY, AES_PLAINTEXT)

print('\nKết luận: ECB lộ hai khối lặp; CBC cho hai khối khác nhau trong mẫu này.')

Bản rõ: b'UIT_LAB_UIT_LAB_UIT_LAB_UIT_LAB_'
Bản rõ: 32 byte; Hex = 5549545f4c41425f5549545f4c41425f5549545f4c41425f5549545f4c41425f
  Khối 1 (16 byte): 5549545f4c41425f5549545f4c41425f
  Khối 2 (16 byte): 5549545f4c41425f5549545f4c41425f
P1 == P2: True

--- AES-ECB ---
Ciphertext dữ liệu: 32 byte; Hex = 02b0647de210da452e3bdcadc415814e02b0647de210da452e3bdcadc415814e
  Khối 1 (16 byte): 02b0647de210da452e3bdcadc415814e
  Khối 2 (16 byte): 02b0647de210da452e3bdcadc415814e
C1 == C2: True
Giải mã khớp bản rõ: True

--- AES-CBC ---
iv: 30303030303030303030303030303030
Ciphertext dữ liệu: 32 byte; Hex = 47747530a46ebd156c1da8763c54be1b46807a7ec513e5e69d9256881a0f0c09
  Khối 1 (16 byte): 47747530a46ebd156c1da8763c54be1b
  Khối 2 (16 byte): 46807a7ec513e5e69d9256881a0f0c09
C1 == C2: False
Giải mã khớp bản rõ: True

Kết luận: ECB lộ hai khối lặp; CBC cho hai khối khác nhau trong mẫu này.


## Các chế độ cổ điển còn lại
CFB-8 xử lý phân đoạn 1 byte; CFB-128 xử lý phân đoạn 16 byte. OFB/CTR tạo dòng khóa, không padding. OPENPGP trong API này là biến thể CFB có tiền tố IV mã hóa, không phải toàn bộ định dạng/giao thức OpenPGP và không tự cung cấp xác thực.

In [ ]:
aes_classic_packets = {}
for aes_mode in ('CFB8', 'CFB128', 'OFB', 'CTR', 'OPENPGP'):
    aes_packet = encrypt_aes(AES_PLAINTEXT, AES_KEY, aes_mode, **demo_options(aes_mode))
    aes_classic_packets[aes_mode] = aes_packet
    show_packet(aes_packet, AES_KEY, AES_PLAINTEXT)

print('\nKhông khối lặp trong mẫu không đồng nghĩa có xác thực.')
aes_ctr_tampered = dict(aes_classic_packets['CTR'])
aes_ctr_tampered['ciphertext'] = flip_first_bit(aes_ctr_tampered['ciphertext'])
aes_ctr_changed_plaintext = decrypt_aes(aes_ctr_tampered, AES_KEY)
print('CTR nhận bản mã bị sửa mà không báo lỗi; bản rõ đã đổi:', aes_ctr_changed_plaintext != AES_PLAINTEXT)


--- AES-CFB8 ---
iv: 30303030303030303030303030303030
Ciphertext dữ liệu: 32 byte; Hex = c6dea3242cfb6add6a411da4024474f41c1be9839020fba9f381b254f2c47a0c
  Khối 1 (16 byte): c6dea3242cfb6add6a411da4024474f4
  Khối 2 (16 byte): 1c1be9839020fba9f381b254f2c47a0c
C1 == C2: False
Giải mã khớp bản rõ: True

--- AES-CFB128 ---
iv: 30303030303030303030303030303030
Ciphertext dữ liệu: 32 byte; Hex = c6f55bf4ba891cc1491a83d7c9767f98c3f3561933409156300ebb2b74aca1c9
  Khối 1 (16 byte): c6f55bf4ba891cc1491a83d7c9767f98
  Khối 2 (16 byte): c3f3561933409156300ebb2b74aca1c9
C1 == C2: False
Giải mã khớp bản rõ: True

--- AES-OFB ---
iv: 30303030303030303030303030303030
Ciphertext dữ liệu: 32 byte; Hex = c6f55bf4ba891cc1491a83d7c9767f98fd0b617b99bebc3ac46d84cb18342642
  Khối 1 (16 byte): c6f55bf4ba891cc1491a83d7c9767f98
  Khối 2 (16 byte): fd0b617b99bebc3ac46d84cb18342642
C1 == C2: False
Giải mã khớp bản rõ: True

--- AES-CTR ---
nonce: 6374722d64656d6f
Ciphertext dữ liệu: 32 byte; Hex = 8f1bc5ee8edd82

## Các chế độ có xác thực
EAX, GCM, CCM, OCB, SIV xuất ciphertext cùng tag. AAD được xác thực nhưng không mã hóa. In nonce, tag, so sánh khối; sửa ciphertext và AAD phải bị từ chối. SIV dùng khóa 32 byte riêng, các chế độ còn lại dùng khóa 16 byte của đề.

In [ ]:
aes_aead_packets = {}
for aes_mode in AES_AUTH_MODES:
    aes_mode_key = demo_key(aes_mode)
    aes_packet = encrypt_aes(AES_PLAINTEXT, aes_mode_key, aes_mode, aad=AES_AAD, **demo_options(aes_mode))
    aes_aead_packets[aes_mode] = aes_packet
    show_packet(aes_packet, aes_mode_key, AES_PLAINTEXT)
    for aes_changed_field in ('ciphertext', 'aad'):
        aes_changed_packet = dict(aes_packet)
        aes_changed_packet[aes_changed_field] = flip_first_bit(aes_changed_packet[aes_changed_field])
        try:
            decrypt_aes(aes_changed_packet, aes_mode_key)
        except ValueError:
            print(f'Sửa {aes_changed_field}: bị từ chối bởi kiểm tra tag.')
        else:
            raise RuntimeError(f'{aes_mode} không phát hiện sửa {aes_changed_field}.')


--- AES-EAX ---
nonce: 6561782d64656d6f2d6e6f6e63653136
tag: 8ea4cf08947c6ee0b8bf8099fc92fce8
AAD: b'NT101|Lab02|Task2.2'
Ciphertext dữ liệu: 32 byte; Hex = d33755ace1fa4bf0b0c2ffebde7475fab78de5345362a503eb851d4bd23f6252
  Khối 1 (16 byte): d33755ace1fa4bf0b0c2ffebde7475fa
  Khối 2 (16 byte): b78de5345362a503eb851d4bd23f6252
C1 == C2: False
Giải mã khớp bản rõ: True
Sửa ciphertext: bị từ chối bởi kiểm tra tag.
Sửa aad: bị từ chối bởi kiểm tra tag.



--- AES-GCM ---
nonce: 67636d2d64656d6f2d30313233
tag: 30ca5b39afefb723bacb74ae50168cf1
AAD: b'NT101|Lab02|Task2.2'
Ciphertext dữ liệu: 32 byte; Hex = a471bf3a6107e742ac97a5ad8d5ff8c5cfdfbcb06cfccfcc5b982244c6c568e7
  Khối 1 (16 byte): a471bf3a6107e742ac97a5ad8d5ff8c5
  Khối 2 (16 byte): cfdfbcb06cfccfcc5b982244c6c568e7
C1 == C2: False
Giải mã khớp bản rõ: True
Sửa ciphertext: bị từ chối bởi kiểm tra tag.
Sửa aad: bị từ chối bởi kiểm tra tag.

--- AES-CCM ---
nonce: 63636d2d64656d6f2d3031
tag: 5f94bf5dc2fa3ba051409a1f2fd0f51a
AAD: b'NT101|Lab02|Task2.2'
Ciphertext dữ liệu: 32 byte; Hex = 19e7bb7c0631ca1acb0ed4bdcf204831e94b743f0ff96dc482a3fc0adb14a286
  Khối 1 (16 byte): 19e7bb7c0631ca1acb0ed4bdcf204831
  Khối 2 (16 byte): e94b743f0ff96dc482a3fc0adb14a286
C1 == C2: False
Giải mã khớp bản rõ: True
Sửa ciphertext: bị từ chối bởi kiểm tra tag.
Sửa aad: bị từ chối bởi kiểm tra tag.

--- AES-OCB ---
nonce: 6f63622d64656d6f2d6e6f6e636531
tag: a2bf0d308227ccf9d3b7b47b1f922da0
AAD: b'NT101|La

## KW/KWP: bọc khóa
KW nhận khóa tối thiểu 16 byte, dài bội số 8; KWP nhận khóa không rỗng dài bất kỳ, có đệm riêng (không phải PKCS#7). Có kiểm tra toàn vẹn tích hợp, mã hóa xác định, không IV/nonce bên ngoài. Thử KW với khóa 16 byte, KWP với khóa 21 byte.

In [ ]:
aes_wrapping_samples = {'KW': bytes(range(16)), 'KWP': bytes(range(21))}
for aes_wrap_mode, aes_key_material in aes_wrapping_samples.items():
    aes_wrapped = wrap_key(aes_key_material, AES_KEY, aes_wrap_mode)
    aes_unwrapped = unwrap_key(aes_wrapped, AES_KEY, aes_wrap_mode)
    print(f'\n--- AES-{aes_wrap_mode} ---')
    print('Số byte khóa cần bọc:', len(aes_key_material))
    print('Khóa đã bọc (Hex):', aes_wrapped.hex())
    print('Mở bọc khớp:', aes_unwrapped == aes_key_material)
    if aes_unwrapped != aes_key_material:
        raise RuntimeError('Mở bọc khóa không khớp.')
    try:
        unwrap_key(flip_first_bit(aes_wrapped), AES_KEY, aes_wrap_mode)
    except ValueError:
        print('Dữ liệu bọc bị sửa: bị từ chối.')
    else:
        raise RuntimeError('Không phát hiện dữ liệu bọc bị sửa.')


--- AES-KW ---
Số byte khóa cần bọc: 16
Khóa đã bọc (Hex): fe2e3a64452eec50bf33dd64775aed6b82c4a683f84dbc8a
Mở bọc khớp: True
Dữ liệu bọc bị sửa: bị từ chối.

--- AES-KWP ---
Số byte khóa cần bọc: 21
Khóa đã bọc (Hex): c9b463d53d496ffc0d2619409d7e5c5aede50c212562e6985929995776fce1c2
Mở bọc khớp: True
Dữ liệu bọc bị sửa: bị từ chối.


## Padding và vai trò IV/nonce
PKCS#7 luôn thêm đệm, kể cả đầu vào đủ khối: 32 byte thành 48 byte. Thử CBC với hai IV khác nhau; thử SIV không nonce để thấy mã hóa xác định. Không dùng các thử nghiệm này làm lý do tái sử dụng IV/nonce trong ứng dụng.

In [ ]:
aes_padded_packet = encrypt_aes(AES_PLAINTEXT, AES_KEY, 'CBC', iv=AES_IV, padding=True)
print('32 byte bản rõ, PKCS#7 thành:', len(aes_padded_packet['ciphertext']), 'byte ciphertext')
print('Bỏ đệm khôi phục bản rõ:', decrypt_aes(aes_padded_packet, AES_KEY) == AES_PLAINTEXT)
aes_short_message = 'An toàn mạng máy tính - UIT'.encode('utf-8')
for aes_mode in ('ECB', 'CBC'):
    aes_short_packet = encrypt_aes(aes_short_message, AES_KEY, aes_mode, padding=True, **demo_options(aes_mode))
    print(f'{aes_mode}: UTF-8 {len(aes_short_message)} byte; ciphertext {len(aes_short_packet["ciphertext"])} byte; giải mã khớp:', decrypt_aes(aes_short_packet, AES_KEY) == aes_short_message)

aes_alternate_iv = bytes(range(16))
aes_cbc_other_iv = encrypt_aes(AES_PLAINTEXT, AES_KEY, 'CBC', iv=aes_alternate_iv)
print('\nCBC, IV thứ nhất:', AES_IV.hex())
print('CBC, IV thứ hai:', aes_alternate_iv.hex())
print('Ciphertext khác khi đổi IV:', aes_cbc_packet['ciphertext'] != aes_cbc_other_iv['ciphertext'])

aes_siv_without_nonce_1 = encrypt_aes(AES_PLAINTEXT, AES_SIV_KEY, 'SIV', aad=AES_AAD)
aes_siv_without_nonce_2 = encrypt_aes(AES_PLAINTEXT, AES_SIV_KEY, 'SIV', aad=AES_AAD)
print('SIV không nonce: cùng bản rõ/AAD/khóa cho cùng ciphertext và tag:', aes_siv_without_nonce_1 == aes_siv_without_nonce_2)
print('SIV không nonce vẫn giải mã/xác thực được:', decrypt_aes(aes_siv_without_nonce_1, AES_SIV_KEY) == AES_PLAINTEXT)

32 byte bản rõ, PKCS#7 thành: 48 byte ciphertext
Bỏ đệm khôi phục bản rõ: True
ECB: UTF-8 32 byte; ciphertext 48 byte; giải mã khớp: True
CBC: UTF-8 32 byte; ciphertext 48 byte; giải mã khớp: True

CBC, IV thứ nhất: 30303030303030303030303030303030
CBC, IV thứ hai: 000102030405060708090a0b0c0d0e0f
Ciphertext khác khi đổi IV: True
SIV không nonce: cùng bản rõ/AAD/khóa cho cùng ciphertext và tag: True
SIV không nonce vẫn giải mã/xác thực được: True


## Tổng hợp và nhận xét
| Chế độ | Padding trong API | Tham số bổ sung | Nhận xét |
|---|---|---|---|
| ECB | Cần khi chưa đủ khối | Không | Hai khối bằng nhau cho ciphertext bằng nhau; lộ mẫu, không xác thực |
| CBC | Cần khi chưa đủ khối | IV 16 byte | Liên kết khối, không lộ mẫu như ECB; không xác thực |
| CFB-8/CFB-128 | Không | IV 16 byte, segment size | Có phản hồi ciphertext; kết quả phụ thuộc phân đoạn, không xác thực |
| OFB | Không | IV 16 byte | Dòng khóa độc lập dữ liệu; lặp IV gây lặp dòng khóa, không xác thực |
| CTR | Không | Nonce và bộ đếm | Dòng khóa theo counter, có thể song song; không được lặp nonce/counter, không xác thực |
| OPENPGP | Không | IV mã hóa 18 byte khi giải mã | Biến thể CFB, tiền tố IV không phải tag xác thực |
| EAX | Không | Nonce, tag, AAD tùy chọn | Mã hóa có xác thực; phát hiện sửa ciphertext/tag/AAD |
| GCM | Không | Nonce (12 byte trong ví dụ), tag | Hiệu quả, hỗ trợ song song; lặp nonce với cùng khóa gây mất an toàn nghiêm trọng |
| CCM | Không | Nonce 7–13 byte, tag | CTR kết hợp CBC-MAC; độ dài nonce giới hạn độ dài thông điệp |
| OCB | Không | Nonce 1–15 byte, tag | Mã hóa có xác thực hiệu quả; nonce phải không lặp |
| SIV | Không | Khóa 32/48/64 byte, nonce tùy chọn, tag | Chống chịu lạm dụng nonce tốt hơn; không nonce thì có tính xác định, lộ việc lặp toàn thông điệp/AAD |
| KW | Đệm riêng không có | Khóa bọc khóa | Bọc khóa dài bội số 8, tối thiểu 16 byte; có kiểm tra toàn vẹn, không dùng thay mã hóa văn bản |
| KWP | Đệm riêng của chuẩn | Khóa bọc khóa | Bọc khóa dài bất kỳ, không rỗng; có kiểm tra toàn vẹn |

### Kết quả chính
- ECB: `C1 = C2 = 02b0647de210da452e3bdcadc415814e`. Mẫu lặp của bản rõ vẫn hiện trong bản mã.
- CBC với IV ASCII đã chọn: `C1 = 47747530a46ebd156c1da8763c54be1b`, `C2 = 46807a7ec513e5e69d9256881a0f0c09`; hai khối khác nhau.
- Các chế độ mã hóa dữ liệu còn lại không có cặp khối lặp trong **mẫu này**. Không suy ra mọi ciphertext đều không bao giờ có khối trùng. Hai khối không đủ để đánh giá thống kê hay tốc độ.
- Mọi chế độ đều giải mã về bản rõ ban đầu khi dùng đúng khóa và tham số. Chỉ nhóm AEAD xác thực ciphertext cùng AAD bằng tag; CTR minh họa dữ liệu bị sửa vẫn được giải mã mà không báo lỗi. KW/KWP xác thực dữ liệu khóa được bọc theo cơ chế riêng.
- IV/nonce thường không cần giữ bí mật, nhưng phải truyền/lưu cùng ciphertext, cùng tag với AEAD. Không tái sử dụng IV/nonce sai yêu cầu của chế độ.
- Padding không tạo tính toàn vẹn; bỏ đệm thành công không chứng minh bản mã chưa bị sửa. Dùng AEAD cho ứng dụng mới, không dùng ECB hay cấu hình khóa cố định trong notebook để bảo vệ dữ liệu thực tế.
- SIV dùng khóa riêng dài gấp đôi, KW/KWP dùng dữ liệu khóa riêng; đây không phải phép đo hiệu năng/công bằng giữa các chế độ.

## Nhập dữ liệu tùy chọn
Đổi `AES_INTERACTIVE = True`, chạy riêng cell dưới; nhập văn bản UTF-8 và chế độ. ECB/CBC tự padding. Sinh khóa, IV/nonce mới ngẫu nhiên mỗi lượt nhập, không lưu khóa. Mặc định `False` để **Run All** không chờ nhập liệu.

In [ ]:
AES_INTERACTIVE = False

if AES_INTERACTIVE:
    try:
        aes_user_mode = input('Chế độ (' + ', '.join(AES_MODES) + '): ').strip().upper()
        if aes_user_mode not in AES_MODES:
            raise ValueError('Chế độ không hợp lệ; dùng CFB8 hoặc CFB128 cho CFB.')
        aes_user_data = input('Nhập bản rõ UTF-8: ').encode('utf-8')
        aes_user_key = get_random_bytes(32 if aes_user_mode == 'SIV' else 16)
        aes_user_options = {}
        if aes_user_mode == 'SIV':
            aes_user_options['nonce'] = get_random_bytes(16)
        aes_user_packet = encrypt_aes(
            aes_user_data, aes_user_key, aes_user_mode,
            padding=aes_user_mode in ('ECB', 'CBC'),
            aad=AES_AAD if aes_user_mode in AES_AUTH_MODES else b'',
            **aes_user_options,
        )
        show_packet(aes_user_packet, aes_user_key, aes_user_data)
        print('Bản rõ khôi phục:', decrypt_aes(aes_user_packet, aes_user_key).decode('utf-8'))
    except ValueError as error:
        print('Lỗi:', error)
else:
    print('Đổi AES_INTERACTIVE = True để nhập dữ liệu.')

Đổi AES_INTERACTIVE = True để nhập dữ liệu.


## Kiểm thử
Vector AES chuẩn, dữ liệu đề, khối lặp, giải mã mọi chế độ, padding, CFB-8/128, OPENPGP, sai khóa/IV/nonce, sửa ciphertext/tag/AAD, KW/KWP và đầu vào không hợp lệ.

In [ ]:
import unittest


class TestAESModes(unittest.TestCase):
    def make_packet(self, mode, data=AES_PLAINTEXT, padding=False):
        return encrypt_aes(data, demo_key(mode), mode, padding=padding,
                           aad=AES_AAD if mode in AES_AUTH_MODES else b'', **demo_options(mode))

    def test_fips_197_vector(self):
        key = bytes.fromhex('000102030405060708090a0b0c0d0e0f')
        plaintext = bytes.fromhex('00112233445566778899aabbccddeeff')
        packet = encrypt_aes(plaintext, key, 'ECB')
        self.assertEqual(packet['ciphertext'].hex(), '69c4e0d86a7b0430d8cdb78070b4c55a')
        self.assertEqual(decrypt_aes(packet, key), plaintext)

    def test_task_lengths_and_blocks(self):
        self.assertEqual(len(AES_PLAINTEXT), 32)
        self.assertEqual(split_blocks(AES_PLAINTEXT), [b'UIT_LAB_UIT_LAB_'] * 2)
        ecb_blocks = split_blocks(self.make_packet('ECB')['ciphertext'])
        cbc_blocks = split_blocks(self.make_packet('CBC')['ciphertext'])
        self.assertEqual(ecb_blocks[0], ecb_blocks[1])
        self.assertNotEqual(cbc_blocks[0], cbc_blocks[1])

    def test_task_ciphertexts(self):
        self.assertEqual(self.make_packet('ECB')['ciphertext'].hex(),
                         '02b0647de210da452e3bdcadc415814e02b0647de210da452e3bdcadc415814e')
        self.assertEqual(self.make_packet('CBC')['ciphertext'].hex(),
                         '47747530a46ebd156c1da8763c54be1b46807a7ec513e5e69d9256881a0f0c09')

    def test_all_modes_round_trip(self):
        samples = [b'', b'X', bytes(range(15)), bytes(range(16)), bytes(range(17)),
                   bytes(range(31)), AES_PLAINTEXT, bytes(range(33)), 'An toàn mạng UIT'.encode('utf-8')]
        for mode in AES_MODES:
            for sample in samples:
                with self.subTest(mode=mode, size=len(sample)):
                    packet = self.make_packet(mode, sample, padding=mode in ('ECB', 'CBC'))
                    self.assertEqual(decrypt_aes(packet, demo_key(mode)), sample)

    def test_ciphertext_lengths_and_tags(self):
        for mode in AES_MODES:
            with self.subTest(mode=mode):
                packet = self.make_packet(mode)
                self.assertEqual(len(packet['ciphertext']), 32)
                if mode in AES_AUTH_MODES:
                    self.assertEqual(len(packet['tag']), 16)
                if mode == 'OPENPGP':
                    self.assertEqual(len(packet['encrypted_iv']), 18)
                    wire = AES.new(AES_KEY, AES.MODE_OPENPGP, iv=AES_IV).encrypt(AES_PLAINTEXT)
                    self.assertEqual(packet['encrypted_iv'] + packet['ciphertext'], wire)

    def test_cfb_segment_sizes(self):
        cfb8 = self.make_packet('CFB8')['ciphertext']
        cfb128 = self.make_packet('CFB128')['ciphertext']
        self.assertNotEqual(cfb8, cfb128)
        self.assertEqual(cfb8, AES.new(AES_KEY, AES.MODE_CFB, iv=AES_IV).encrypt(AES_PLAINTEXT))
        self.assertEqual(cfb128, AES.new(AES_KEY, AES.MODE_CFB, iv=AES_IV, segment_size=128).encrypt(AES_PLAINTEXT))

    def test_pkcs7_always_adds_padding(self):
        for mode in ('ECB', 'CBC'):
            for length in (0, 1, 15, 16, 17, 31, 32, 33):
                with self.subTest(mode=mode, length=length):
                    data = bytes(range(length))
                    packet = self.make_packet(mode, data, padding=True)
                    self.assertEqual(len(packet['ciphertext']), (length // 16 + 1) * 16)
                    self.assertEqual(decrypt_aes(packet, AES_KEY), data)

    def test_generated_iv_and_nonce(self):
        for mode in AES_MODES:
            if mode in ('ECB', 'SIV'):
                continue
            with self.subTest(mode=mode):
                first = encrypt_aes(AES_PLAINTEXT, AES_KEY, mode)
                second = encrypt_aes(AES_PLAINTEXT, AES_KEY, mode)
                self.assertNotEqual(first['ciphertext'], second['ciphertext'])
                self.assertEqual(decrypt_aes(first, AES_KEY), AES_PLAINTEXT)
                self.assertEqual(decrypt_aes(second, AES_KEY), AES_PLAINTEXT)

    def test_siv_without_nonce(self):
        first = encrypt_aes(AES_PLAINTEXT, AES_SIV_KEY, 'SIV', aad=AES_AAD)
        second = encrypt_aes(AES_PLAINTEXT, AES_SIV_KEY, 'SIV', aad=AES_AAD)
        self.assertNotIn('nonce', first)
        self.assertEqual(first, second)
        self.assertEqual(decrypt_aes(first, AES_SIV_KEY), AES_PLAINTEXT)

    def test_all_library_modes_covered(self):
        supported = {getattr(AES, name) for name in dir(AES) if name.startswith('MODE_')}
        covered = set(AES_MODES.values()) | {AES.MODE_KW, AES.MODE_KWP}
        self.assertEqual(covered, supported)

In [ ]:
class TestAESIntegrity(unittest.TestCase):
    def test_aead_rejects_modified_fields(self):
        for mode in AES_AUTH_MODES:
            key = demo_key(mode)
            packet = encrypt_aes(AES_PLAINTEXT, key, mode, aad=AES_AAD, **demo_options(mode))
            for field in ('ciphertext', 'tag', 'aad', 'nonce'):
                with self.subTest(mode=mode, field=field):
                    changed = dict(packet)
                    changed[field] = flip_first_bit(changed[field])
                    with self.assertRaises(ValueError):
                        decrypt_aes(changed, key)
            with self.subTest(mode=mode, field='key'):
                with self.assertRaises(ValueError):
                    decrypt_aes(packet, flip_first_bit(key))

    def test_classic_modes_do_not_authenticate(self):
        for mode in ('ECB', 'CBC', 'CFB8', 'CFB128', 'OFB', 'CTR', 'OPENPGP'):
            with self.subTest(mode=mode):
                packet = encrypt_aes(AES_PLAINTEXT, AES_KEY, mode, **demo_options(mode))
                packet['ciphertext'] = flip_first_bit(packet['ciphertext'])
                self.assertNotEqual(decrypt_aes(packet, AES_KEY), AES_PLAINTEXT)

    def test_kw_rfc3394_vector(self):
        wrapping_key = bytes.fromhex('000102030405060708090a0b0c0d0e0f')
        material = bytes.fromhex('00112233445566778899aabbccddeeff')
        wrapped = wrap_key(material, wrapping_key, 'KW')
        self.assertEqual(wrapped.hex(), '1fa68b0a8112b447aef34bd8fb5a7b829d3e862371d2cfe5')
        self.assertEqual(unwrap_key(wrapped, wrapping_key, 'KW'), material)

    def test_wrapping_round_trip_and_integrity(self):
        for mode, lengths in (('KW', (16, 24, 32)), ('KWP', (1, 7, 8, 9, 16, 21, 32))):
            for length in lengths:
                with self.subTest(mode=mode, length=length):
                    material = bytes(range(length))
                    wrapped = wrap_key(material, AES_KEY, mode)
                    self.assertEqual(unwrap_key(wrapped, AES_KEY, mode), material)
                    self.assertEqual(wrap_key(material, AES_KEY, mode), wrapped)
                    with self.assertRaises(ValueError):
                        unwrap_key(flip_first_bit(wrapped), AES_KEY, mode)
                    with self.assertRaises(ValueError):
                        unwrap_key(wrapped, flip_first_bit(AES_KEY), mode)

    def test_wrapping_rejects_invalid_lengths(self):
        for length in (0, 1, 8, 15, 17):
            with self.subTest(length=length):
                with self.assertRaises(ValueError):
                    wrap_key(bytes(length), AES_KEY, 'KW')
        with self.assertRaises(ValueError):
            wrap_key(b'', AES_KEY, 'KWP')
        with self.assertRaises(ValueError):
            wrap_key(bytes(16), AES_KEY, 'CBC')

    def test_invalid_inputs(self):
        for invalid_key in (b'', b'short', bytes(15), bytes(17), bytes(31)):
            with self.assertRaises(ValueError):
                encrypt_aes(AES_PLAINTEXT, invalid_key, 'ECB')
        with self.assertRaises(ValueError):
            encrypt_aes(AES_PLAINTEXT, AES_KEY, 'SIV')
        for invalid_iv in (b'', bytes(15), bytes(17)):
            with self.assertRaises(ValueError):
                encrypt_aes(AES_PLAINTEXT, AES_KEY, 'CBC', iv=invalid_iv)
        for mode, nonce in (('CCM', bytes(6)), ('CCM', bytes(14)), ('OCB', bytes(16)),
                            ('CTR', bytes(16)), ('GCM', b'')):
            with self.subTest(mode=mode):
                with self.assertRaises(ValueError):
                    encrypt_aes(AES_PLAINTEXT, AES_KEY, mode, nonce=nonce)
        for data in (b'X', bytes(15), bytes(17)):
            for mode in ('ECB', 'CBC'):
                with self.assertRaises(ValueError):
                    encrypt_aes(data, AES_KEY, mode)
        for options in ({'mode': 'UNKNOWN'}, {'mode': 'CTR', 'padding': True},
                        {'mode': 'ECB', 'iv': AES_IV}, {'mode': 'CBC', 'nonce': b'nonce'},
                        {'mode': 'ECB', 'aad': b'header'}, {'mode': 'ECB', 'padding': 1}):
            with self.assertRaises(ValueError):
                encrypt_aes(AES_PLAINTEXT, AES_KEY, **options)
        with self.assertRaises(ValueError):
            encrypt_aes('Không phải bytes', AES_KEY, 'ECB')
        for block_size in (0, -1, True, 1.5):
            with self.assertRaises(ValueError):
                split_blocks(AES_PLAINTEXT, block_size)

In [ ]:
aes_test_suite = unittest.TestSuite([
    unittest.defaultTestLoader.loadTestsFromTestCase(TestAESModes),
    unittest.defaultTestLoader.loadTestsFromTestCase(TestAESIntegrity),
])
aes_test_result = unittest.TextTestRunner(verbosity=2).run(aes_test_suite)
if not aes_test_result.wasSuccessful():
    raise RuntimeError('Kiểm thử AES thất bại.')

test_all_library_modes_covered (__main__.TestAESModes.test_all_library_modes_covered) ... 

ok


test_all_modes_round_trip (__main__.TestAESModes.test_all_modes_round_trip) ... 

ok


test_cfb_segment_sizes (__main__.TestAESModes.test_cfb_segment_sizes) ... 

...

test_wrapping_rejects_invalid_lengths (__main__.TestAESIntegrity.test_wrapping_rejects_invalid_lengths) ... 

ok


test_wrapping_round_trip_and_integrity (__main__.TestAESIntegrity.test_wrapping_round_trip_and_integrity) ... 

ok


----------------------------------------------------------------------
Ran 16 tests in 0.340s

OK


## Tài liệu tham khảo
- AES API: https://www.pycryptodome.org/src/cipher/aes.
- Các chế độ cổ điển: https://www.pycryptodome.org/src/cipher/classic.
- AEAD, KW/KWP: https://www.pycryptodome.org/src/cipher/modern.
- Padding: https://www.pycryptodome.org/src/util/util#module-Crypto.Util.Padding.
- Vector AES-128: FIPS 197; https://csrc.nist.gov/pubs/fips/197/final.
- Vector bọc khóa AES-KW: RFC 3394, mục 4.1; https://www.rfc-editor.org/rfc/rfc3394#section-4.1.